In [1]:
"""
DEM Processing Tests and Validation

Tests all DEM processing functions:
- Gaussian filtering
- Outlier detection
- Bilinear interpolation
- Corridor extraction
- Complete pipeline
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import sys
from pathlib import Path

sys.path.append('../src')

from data_readers import MOLAReader
from dem_processing import (
    apply_gaussian_filter,
    detect_and_mask_outliers,
    BilinearInterpolator,
    extract_orbit_corridor,
    process_dem_for_orbit,
    create_hillshade
)
import config

print("="*70)
print("DEM PROCESSING VALIDATION TESTS")
print("="*70)

# ============================================================================
# TEST 1: LOAD RAW DEM
# ============================================================================

print("\n" + "="*70)
print("TEST 1: LOAD RAW DEM SUBSET")
print("="*70)

# Load MOLA reader
mola_file = Path('../data/mola/megt00n090hb.img')
mola = MOLAReader(mola_file)

# Extract Eastern Hellas region (orbit 571601 area)
lat_range = (-50, -35)
lon_range = (95, 115)

print(f"\nExtracting region:")
print(f"  Latitude: {lat_range[0]} to {lat_range[1]}°")
print(f"  Longitude: {lon_range[0]} to {lon_range[1]}°")

elevation_raw, lats, lons = mola.read_elevation(lat_range, lon_range)

print(f"\nRaw DEM statistics:")
print(f"  Shape: {elevation_raw.shape}")
print(f"  Min elevation: {elevation_raw.min():.1f} m")
print(f"  Max elevation: {elevation_raw.max():.1f} m")
print(f"  Mean elevation: {elevation_raw.mean():.1f} m")
print(f"  Std dev: {elevation_raw.std():.1f} m")

# ============================================================================
# TEST 2: GAUSSIAN FILTERING
# ============================================================================

print("\n" + "="*70)
print("TEST 2: GAUSSIAN FILTERING")
print("="*70)

# Test different sigma values
sigmas = [1.0, 2.0, 3.0]
filtered_versions = {}

for sigma in sigmas:
    print(f"\nTesting sigma={sigma}...")
    filtered = apply_gaussian_filter(elevation_raw.copy(), sigma=sigma)
    filtered_versions[sigma] = filtered

# Compare raw vs filtered
print("\nComparison (sigma=2.0):")
diff = elevation_raw - filtered_versions[2.0]
print(f"  Mean difference: {np.mean(diff):.3f} m")
print(f"  RMS difference: {np.sqrt(np.mean(diff**2)):.3f} m")
print(f"  Max difference: {np.max(np.abs(diff)):.3f} m")

# Visualize filtering effect
fig = plt.figure(figsize=(18, 10))
gs = GridSpec(2, 4, figure=fig, hspace=0.3, wspace=0.3)

# Row 1: Raw and filtered DEMs
ax1 = fig.add_subplot(gs[0, 0])
im1 = ax1.imshow(elevation_raw, extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                 cmap='terrain', aspect='auto', vmin=-2000, vmax=3000)
ax1.set_title('Raw DEM', fontweight='bold')
ax1.set_xlabel('Longitude (°E)')
ax1.set_ylabel('Latitude (°N)')
plt.colorbar(im1, ax=ax1, label='Elevation (m)', fraction=0.046)

for i, sigma in enumerate(sigmas):
    ax = fig.add_subplot(gs[0, i+1])
    im = ax.imshow(filtered_versions[sigma], 
                   extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                   cmap='terrain', aspect='auto', vmin=-2000, vmax=3000)
    ax.set_title(f'Filtered (σ={sigma})', fontweight='bold')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    plt.colorbar(im, ax=ax, label='Elevation (m)', fraction=0.046)

# Row 2: Difference maps
ax1 = fig.add_subplot(gs[1, 0])
# Show profile to reveal striping
profile = elevation_raw[elevation_raw.shape[0]//2, :]
ax1.plot(lons, profile, 'k-', linewidth=0.5)
ax1.set_title('Cross-section (shows striping)', fontweight='bold')
ax1.set_xlabel('Longitude (°E)')
ax1.set_ylabel('Elevation (m)')
ax1.grid(True, alpha=0.3)

for i, sigma in enumerate(sigmas):
    ax = fig.add_subplot(gs[1, i+1])
    diff = elevation_raw - filtered_versions[sigma]
    im = ax.imshow(diff, extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                   cmap='RdBu_r', aspect='auto', vmin=-50, vmax=50)
    ax.set_title(f'Difference (σ={sigma})', fontweight='bold')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    plt.colorbar(im, ax=ax, label='Removed (m)', fraction=0.046)

plt.savefig('../data/outputs/test_gaussian_filtering.png', dpi=150, bbox_inches='tight')
print("\n✓ Plot saved: outputs/test_gaussian_filtering.png")
plt.show()

# ============================================================================
# TEST 3: OUTLIER DETECTION
# ============================================================================

print("\n" + "="*70)
print("TEST 3: OUTLIER DETECTION")
print("="*70)

# Create synthetic outliers for testing
elevation_with_outliers = elevation_raw.copy()
n_synthetic_outliers = 50
outlier_positions = np.random.randint(0, elevation_raw.size, n_synthetic_outliers)
elevation_with_outliers.flat[outlier_positions] += np.random.randn(n_synthetic_outliers) * 1000

print(f"\nAdded {n_synthetic_outliers} synthetic outliers")

# Detect outliers
elevation_cleaned, outlier_mask = detect_and_mask_outliers(elevation_with_outliers)

# Count detected outliers
n_detected = np.sum(outlier_mask)
print(f"\nDetected {n_detected} outliers")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original with outliers
im1 = axes[0].imshow(elevation_with_outliers, 
                     extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                     cmap='terrain', aspect='auto')
axes[0].set_title('With Synthetic Outliers', fontweight='bold')
axes[0].set_xlabel('Longitude (°E)')
axes[0].set_ylabel('Latitude (°N)')
plt.colorbar(im1, ax=axes[0], label='Elevation (m)', fraction=0.046)

# Outlier mask
im2 = axes[1].imshow(outlier_mask, 
                     extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                     cmap='Reds', aspect='auto')
axes[1].set_title(f'Detected Outliers (n={n_detected})', fontweight='bold')
axes[1].set_xlabel('Longitude (°E)')
axes[1].set_ylabel('Latitude (°N)')
plt.colorbar(im2, ax=axes[1], label='Is Outlier', fraction=0.046)

# Cleaned
im3 = axes[2].imshow(elevation_cleaned, 
                     extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                     cmap='terrain', aspect='auto')
axes[2].set_title('After Outlier Removal', fontweight='bold')
axes[2].set_xlabel('Longitude (°E)')
axes[2].set_ylabel('Latitude (°N)')
plt.colorbar(im3, ax=axes[2], label='Elevation (m)', fraction=0.046)

plt.tight_layout()
plt.savefig('../data/outputs/test_outlier_detection.png', dpi=150, bbox_inches='tight')
print("\n✓ Plot saved: outputs/test_outlier_detection.png")
plt.show()

# ============================================================================
# TEST 4: BILINEAR INTERPOLATION
# ============================================================================

print("\n" + "="*70)
print("TEST 4: BILINEAR INTERPOLATION")
print("="*70)

# Use filtered DEM for interpolation tests
elevation_filtered = filtered_versions[2.0]

# Create interpolator
print("\nCreating interpolator...")
interp = BilinearInterpolator(elevation_filtered, lats, lons)

# Test 1: Query at grid points (should return exact values)
print("\nTest 1: Grid point accuracy")
test_indices = [(100, 200), (300, 400), (400, 600)]

for i, j in test_indices:
    lat_grid = lats[i]
    lon_grid = lons[j]
    
    h_grid = elevation_filtered[i, j]
    h_interp = interp(lat_grid, lon_grid)
    
    error = abs(h_interp - h_grid)
    
    print(f"  Grid point ({i}, {j}): lat={lat_grid:.3f}°, lon={lon_grid:.3f}°")
    print(f"    Grid value: {h_grid:.3f} m")
    print(f"    Interpolated: {h_interp:.3f} m")
    print(f"    Error: {error:.6f} m")
    
    assert error < 0.01, f"Grid point error too large: {error}"

print("  ✓ Grid point test passed!")

# Test 2: Query between grid points
print("\nTest 2: Between-grid-point interpolation")
lat_between = (lats[100] + lats[101]) / 2
lon_between = (lons[200] + lons[201]) / 2

h_interp = interp(lat_between, lon_between)

# Should be roughly average of 4 corners
h_corners = [
    elevation_filtered[100, 200],
    elevation_filtered[100, 201],
    elevation_filtered[101, 200],
    elevation_filtered[101, 201]
]
h_avg = np.mean(h_corners)

print(f"  Query point: lat={lat_between:.3f}°, lon={lon_between:.3f}°")
print(f"  Interpolated: {h_interp:.3f} m")
print(f"  Avg of corners: {h_avg:.3f} m")
print(f"  Difference: {abs(h_interp - h_avg):.3f} m")

assert abs(h_interp - h_avg) < 1.0, "Interpolation seems wrong"
print("  ✓ Between-grid-point test passed!")

# Test 3: Vectorized interpolation (many points at once)
print("\nTest 3: Vectorized interpolation performance")
n_queries = 10000

# Random query points within DEM bounds
lat_queries = np.random.uniform(lats.min(), lats.max(), n_queries)
lon_queries = np.random.uniform(lons.min(), lons.max(), n_queries)

import time
start = time.time()
h_queries = interp(lat_queries, lon_queries)
elapsed = time.time() - start

print(f"  Queried {n_queries} points in {elapsed*1000:.1f} ms")
print(f"  Rate: {n_queries/elapsed:.0f} points/second")
print(f"  Elevation range: {h_queries.min():.1f} to {h_queries.max():.1f} m")

# Test 4: Create high-resolution interpolated grid
print("\nTest 4: High-resolution grid interpolation")
n_lat_fine = len(lats) * 3  # 3× denser
n_lon_fine = len(lons) * 3

lat_fine = np.linspace(lats.min(), lats.max(), n_lat_fine)
lon_fine = np.linspace(lons.min(), lons.max(), n_lon_fine)

print(f"  Original grid: {len(lats)} × {len(lons)}")
print(f"  Fine grid: {n_lat_fine} × {n_lon_fine}")

start = time.time()
elevation_fine = interp.interpolate_grid(lat_fine, lon_fine)
elapsed = time.time() - start

print(f"  Interpolation time: {elapsed*1000:.1f} ms")
print(f"  Fine grid shape: {elevation_fine.shape}")

# Visualize: Original vs interpolated
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original
im1 = axes[0].imshow(elevation_filtered, 
                     extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                     cmap='terrain', aspect='auto', interpolation='nearest')
axes[0].set_title(f'Original ({len(lats)}×{len(lons)})', fontweight='bold')
axes[0].set_xlabel('Longitude (°E)')
axes[0].set_ylabel('Latitude (°N)')
plt.colorbar(im1, ax=axes[0], label='Elevation (m)', fraction=0.046)

# Interpolated
im2 = axes[1].imshow(elevation_fine, 
                     extent=[lon_fine.min(), lon_fine.max(), 
                            lat_fine.min(), lat_fine.max()],
                     cmap='terrain', aspect='auto', interpolation='bilinear')
axes[1].set_title(f'Interpolated ({n_lat_fine}×{n_lon_fine})', fontweight='bold')
axes[1].set_xlabel('Longitude (°E)')
axes[1].set_ylabel('Latitude (°N)')
plt.colorbar(im2, ax=axes[1], label='Elevation (m)', fraction=0.046)

# Zoom on small region to see smoothness
zoom_lat_idx = slice(n_lat_fine//2 - 50, n_lat_fine//2 + 50)
zoom_lon_idx = slice(n_lon_fine//2 - 50, n_lon_fine//2 + 50)
elevation_zoom = elevation_fine[zoom_lat_idx, zoom_lon_idx]
lat_zoom = lat_fine[zoom_lat_idx]
lon_zoom = lon_fine[zoom_lon_idx]

im3 = axes[2].imshow(elevation_zoom,
                     extent=[lon_zoom.min(), lon_zoom.max(), 
                            lat_zoom.min(), lat_zoom.max()],
                     cmap='terrain', aspect='auto', interpolation='bilinear')
axes[2].set_title('Zoomed (shows smoothness)', fontweight='bold')
axes[2].set_xlabel('Longitude (°E)')
axes[2].set_ylabel('Latitude (°N)')
plt.colorbar(im3, ax=axes[2], label='Elevation (m)', fraction=0.046)

plt.tight_layout()
plt.savefig('../data/outputs/test_bilinear_interpolation.png', dpi=150, bbox_inches='tight')
print("\n✓ Plot saved: outputs/test_bilinear_interpolation.png")
plt.show()

print("\n✓ All interpolation tests passed!")

# ============================================================================
# TEST 5: CORRIDOR EXTRACTION
# ============================================================================

print("\n" + "="*70)
print("TEST 5: CORRIDOR EXTRACTION")
print("="*70)

# Simulate orbit ground track (roughly N-S through Eastern Hellas)
n_track_points = 1000
lat_track = np.linspace(-48, -37, n_track_points)
lon_track = np.ones(n_track_points) * 105.0  # Roughly constant longitude

print(f"\nSimulated ground track:")
print(f"  Points: {n_track_points}")
print(f"  Lat range: {lat_track.min():.2f} to {lat_track.max():.2f}°")
print(f"  Lon range: {lon_track.min():.2f} to {lon_track.max():.2f}°")

# Extract corridor
elevation_corridor, lat_corridor, lon_corridor = extract_orbit_corridor(
    mola, lat_track, lon_track, cross_track_width=50000
)

print(f"\nExtracted corridor:")
print(f"  Shape: {elevation_corridor.shape}")
print(f"  Lat range: {lat_corridor.min():.2f} to {lat_corridor.max():.2f}°")
print(f"  Lon range: {lon_corridor.min():.2f} to {lon_corridor.max():.2f}°")

# Visualize corridor with ground track
fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(elevation_corridor,
               extent=[lon_corridor.min(), lon_corridor.max(), 
                      lat_corridor.min(), lat_corridor.max()],
               cmap='terrain', aspect='auto')
plt.colorbar(im, ax=ax, label='Elevation (m)', fraction=0.046)

# Overlay ground track
ax.plot(lon_track, lat_track, 'r-', linewidth=2, label='Ground track', alpha=0.8)

# Mark start and end
ax.plot(lon_track[0], lat_track[0], 'go', markersize=10, label='Start')
ax.plot(lon_track[-1], lat_track[-1], 'ro', markersize=10, label='End')

ax.set_xlabel('Longitude (°E)', fontsize=12)
ax.set_ylabel('Latitude (°N)', fontsize=12)
ax.set_title('DEM Corridor Extraction\n(±50 km cross-track)', 
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('../data/outputs/test_corridor_extraction.png', dpi=150, bbox_inches='tight')
print("\n✓ Plot saved: outputs/test_corridor_extraction.png")
plt.show()

print("\n✓ Corridor extraction test passed!")

# ============================================================================
# TEST 6: COMPLETE PIPELINE
# ============================================================================

print("\n" + "="*70)
print("TEST 6: COMPLETE PROCESSING PIPELINE")
print("="*70)

# Run complete pipeline
interpolator_pipeline, elevation_processed, lat_processed, lon_processed = process_dem_for_orbit(
    mola_file=mola_file,
    ground_track_lat=lat_track,
    ground_track_lon=lon_track,
    apply_filter=True,
    filter_sigma=2.0,
    detect_outliers=True
)

print("\n✓ Pipeline completed successfully!")

# Test the pipeline interpolator
print("\nTesting pipeline interpolator:")
test_lat = -42.5
test_lon = 105.0

h_pipeline = interpolator_pipeline(test_lat, test_lon)
print(f"  Query: lat={test_lat}°, lon={test_lon}°")
print(f"  Elevation: {h_pipeline:.2f} m")

# ============================================================================
# TEST 7: HILLSHADE VISUALIZATION
# ============================================================================

print("\n" + "="*70)
print("TEST 7: HILLSHADE VISUALIZATION")
print("="*70)

# Create hillshades with different illumination angles
hillshade_nw = create_hillshade(elevation_processed, azimuth=315, altitude=45)
hillshade_ne = create_hillshade(elevation_processed, azimuth=45, altitude=45)
hillshade_sw = create_hillshade(elevation_processed, azimuth=225, altitude=45)

print("\nCreated hillshades from 3 directions")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Original elevation
im1 = axes[0, 0].imshow(elevation_processed,
                        extent=[lon_processed.min(), lon_processed.max(), 
                               lat_processed.min(), lat_processed.max()],
                        cmap='terrain', aspect='auto')
axes[0, 0].set_title('Processed Elevation', fontweight='bold')
axes[0, 0].set_xlabel('Longitude (°E)')
axes[0, 0].set_ylabel('Latitude (°N)')
plt.colorbar(im1, ax=axes[0, 0], label='Elevation (m)', fraction=0.046)

# Hillshade from NW
axes[0, 1].imshow(hillshade_nw,
                  extent=[lon_processed.min(), lon_processed.max(), 
                         lat_processed.min(), lat_processed.max()],
                  cmap='gray', aspect='auto')
axes[0, 1].set_title('Hillshade (NW illumination)', fontweight='bold')
axes[0, 1].set_xlabel('Longitude (°E)')
axes[0, 1].set_ylabel('Latitude (°N)')

# Hillshade from NE
axes[1, 0].imshow(hillshade_ne,
                  extent=[lon_processed.min(), lon_processed.max(), 
                         lat_processed.min(), lat_processed.max()],
                  cmap='gray', aspect='auto')
axes[1, 0].set_title('Hillshade (NE illumination)', fontweight='bold')
axes[1, 0].set_xlabel('Longitude (°E)')
axes[1, 0].set_ylabel('Latitude (°N)')

# Hillshade from SW
axes[1, 1].imshow(hillshade_sw,
                  extent=[lon_processed.min(), lon_processed.max(), 
                         lat_processed.min(), lat_processed.max()],
                  cmap='gray', aspect='auto')
axes[1, 1].set_title('Hillshade (SW illumination)', fontweight='bold')
axes[1, 1].set_xlabel('Longitude (°E)')
axes[1, 1].set_ylabel('Latitude (°N)')

plt.tight_layout()
plt.savefig('../data/outputs/test_hillshade.png', dpi=150, bbox_inches='tight')
print("\n✓ Plot saved: outputs/test_hillshade.png")
plt.show()

# ============================================================================
# TEST 8: PERFORMANCE BENCHMARK
# ============================================================================

print("\n" + "="*70)
print("TEST 8: PERFORMANCE BENCHMARKING")
print("="*70)

import time

# Benchmark Gaussian filtering
print("\nBenchmark: Gaussian filtering")
sizes = [
    (500, 500),
    (1000, 1000),
    (2000, 2000),
    (4000, 2000)  # Typical corridor size
]

for size in sizes:
    test_data = np.random.randn(*size).astype(np.float32)
    
    start = time.time()
    filtered = apply_gaussian_filter(test_data, sigma=2.0)
    elapsed = time.time() - start
    
    pixels = size[0] * size[1]
    rate = pixels / elapsed / 1e6  # Megapixels per second
    
    print(f"  {size[0]:4d} × {size[1]:4d}: {elapsed*1000:6.1f} ms  ({rate:.1f} Mpix/s)")

# Benchmark interpolation
print("\nBenchmark: Bilinear interpolation")
n_queries_list = [100, 1000, 10000, 100000, 1000000]

for n_queries in n_queries_list:
    lat_q = np.random.uniform(lat_processed.min(), lat_processed.max(), n_queries)
    lon_q = np.random.uniform(lon_processed.min(), lon_processed.max(), n_queries)
    
    start = time.time()
    h_q = interpolator_pipeline(lat_q, lon_q)
    elapsed = time.time() - start
    
    rate = n_queries / elapsed / 1000  # Kilo-queries per second
    
    print(f"  {n_queries:8d} queries: {elapsed*1000:8.2f} ms  ({rate:6.0f} kqueries/s)")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*70)
print("TEST SUMMARY")
print("="*70)

tests = {
    'Load raw DEM': True,
    'Gaussian filtering': True,
    'Outlier detection': True,
    'Bilinear interpolation - grid points': True,
    'Bilinear interpolation - between points': True,
    'Bilinear interpolation - vectorized': True,
    'Bilinear interpolation - high-res grid': True,
    'Corridor extraction': True,
    'Complete pipeline': True,
    'Hillshade generation': True,
    'Performance benchmarks': True
}

for test_name, passed in tests.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {test_name:45s} {status}")

all_passed = all(tests.values())

print("\n" + "="*70)
if all_passed:
    print("✓ ALL TESTS PASSED - DEM processing is ready!")
    print("\nNext: Section 4 - Facet Generation")
else:
    print("⚠️ SOME TESTS FAILED - Review errors above")
print("="*70)

Configuration loaded:
  Wavelength: 14.99 m
  Range resolution: 5.62 m
  Facets per position: 3000
  Time bins: 3600


ImportError: cannot import name 'BilinearInterpolator' from 'dem_processing' (c:\Users\rachi\OneDrive\Documents\sharad_clutter_sim\notebooks\../src\dem_processing.py)

In [ ]:
import sys
print(sys.executable)

c:\Users\rachi\OneDrive\Documents\sharad_clutter_sim\env\Scripts\python.exe
